# NetraTrust — one Kaggle T4 run, two real artifacts

This notebook is the **only** place either real model artifact is produced:

| # | Artifact | What it is |
|---|---|---|
| 1 | `convnext_tiny_aptos.pt` | the DR backbone, fine-tuned on APTOS, marked `trained: true` |
| 2 | `quality_head_eyeq.pkl`  | the QuickQual gate head, fitted on **real human EyeQ labels** |

Artifact 2 replaces `data/quality_head_synthetic.pkl`, whose labels came from
`cv2.GaussianBlur` and which no human ever looked at. The two are never pooled
and never share a filename (`.claude/rules/ml.md`).

---

## Before you press Run All

### 1. Add Input — three datasets

| Attach | Why | What the notebook does with it |
|---|---|---|
| **your `netratrust-code` dataset** | Lane C's code + the frozen splits | built by `python scripts/package_for_kaggle.py`; upload `kaggle_payload.zip` as a Dataset |
| **an APTOS 2019 dataset** | the DR corpus | e.g. `aptos2019-blindness-detection` (competition) or `mariaherrerot/aptos2019` |
| **an EyePACS mirror** | the images EyeQ's labels point at | e.g. `tanlikesmath/diabetic-retinopathy-resized`. **Do not download it** — attach the mount |

Nothing is hardcoded to a mount path. The notebook finds the APTOS label CSVs
by their `id_code,diagnosis` header, the APTOS image directory by asking which
one actually holds the ids in the frozen splits, and the EyePACS directory by
the `<patient>_<left|right>` filename shape. Every candidate it found is
printed, so you can see which one it picked.

### 2. Settings

* **Accelerator: GPU T4 x2.** On CPU this run is pointless; section 1 stops.
* **Internet: On.** Only to fetch EyeQ's two label CSVs (~700 KiB total) from
  `github.com/HzFu/EyeQ`. **No images are ever downloaded.**
* Persistence: off. Everything needed is written to `/kaggle/working`.

### 3. Run order

Run All, top to bottom. It is one pass, in this order:

| § | Does | Rough T4 time |
|---|---|---|
| 1 | mounts, versions, device | seconds |
| 2 | grow the frozen splits to the full APTOS corpus | < 1 min |
| 3 | retained fractions — **stops if any is below 0.95** | 1–3 min |
| 4 | **train the DR backbone** | 1.5–3 h |
| 5 | calibration QWK per epoch, leak alarm | seconds |
| 6 | conformal abstention, fitted on calibration | 5–10 min |
| 7 | EyeQ labels → mounted EyePACS → **fit the real quality head** | 20–50 min |
| 8 | what is real, what is synthetic, where everything landed | seconds |

Sections 4 and 7 are independent. If you are short of time, lower `EPOCHS`
and `EYEQ_PER_CLASS` in section 1 rather than skipping a section — a run that
produces one artifact is not the run this notebook is for.

### 4. Outputs — all under `/kaggle/working/artifacts/`

```
convnext_tiny_aptos.pt       the DR checkpoint          (trained: true)
dr_run.json                  its metrics and provenance
quality_head_eyeq.pkl        the REAL quality head
quality_head_eyeq.json       its metrics and provenance
conformal_calibration.json   abstention coverage on the calibration fold
run_summary.json             real vs synthetic, in one place
manifest_full.csv            the grown manifest        (commit this)
splits_full/                 the grown frozen splits   (commit these)
```

Download them from the notebook's **Output** tab when the run finishes.

### 5. It stops rather than continuing, when

* the accelerator is not a GPU;
* growing the splits would move, drop or relabel **any** image that already
  had a fold (`SplitDrift`) — the splits are frozen and growth must be
  provably additive;
* a retained fraction for train or calibration is below **0.95** — the
  experiment would be smaller than it looks;
* calibration QWK comes out **above 0.95** — SPEC.md §2: that means a leaked
  split, not a good model. Find the leak.

### 6. Rules this notebook does not break

1. Splits come from the frozen files. Section 2 **grows** them at the same
   seed and proves nothing moved; it never re-shuffles.
2. **The test split is never collected here.** It is touched exactly once, at
   the very end, outside this notebook.
3. Every metric comes from `netra/eval/metrics.py`. Nothing is computed inline.
4. Preprocessing is background removal + resize only. No CLAHE, no Ben Graham —
   a controlled ablation put both *below* plain crop+resize (0.8260 and 0.8238
   against 0.8310).
5. Retained fractions are printed at every filter.
6. Real and synthetic results stay in separate tables.


## 1. Mounts, versions, device

Nothing below is a hardcoded Kaggle path. If the payload is not found under
`/kaggle/input`, this falls back to a local clone so the notebook can be
opened and read off-Kaggle without erroring on cell 1.


In [ ]:
import json
import pathlib
import sys
import time

import torch

# ---- knobs. Lower these two if you are short of session time. -------------
EPOCHS = 12
BATCH_SIZE = 16  # 512px on a T4; drop to 8 if it OOMs
LEARNING_RATE = 3e-4
#: Cap per EyeQ quality class per fold. None uses all 28,792 labelled images.
EYEQ_PER_CLASS = 1500
EYEQ_BATCH_SIZE = 32
#: False trains on the committed 366-image splits instead of growing them.
GROW_SPLITS = True
#: The hard stop from .claude/rules/ml.md, applied to the APTOS folds.
MIN_RETAINED = 0.95
#: SPEC.md section 2. Above this is a leaked split, not a good model.
LEAK_ALARM_QWK = 0.95
ALLOW_CPU = False
# ---------------------------------------------------------------------------

KAGGLE_INPUT = pathlib.Path("/kaggle/input")


class Stop(RuntimeError):
    """Raised where continuing would produce a number we could not defend.

    A named exception rather than SystemExit: under Run All this has to halt
    the notebook visibly, and it has to be greppable - every condition in
    section 5 of the run book raises this and nothing else does.
    """


def find_payload(search_root):
    """The attached code dataset: whatever directory holds netra/models/train.py."""
    if not search_root.exists():
        return None
    for depth in ("*", "*/*", "*/*/*", "*/*/*/*"):
        for hit in sorted(search_root.glob(f"{depth}/netra/models/train.py")):
            return hit.parents[2]
    # last resort: recursive search at any depth (handles /kaggle/input/datasets/<owner>/<slug>/)
    for hit in sorted(search_root.rglob("netra/models/train.py")):
        return hit.parents[2]
    return None


PAYLOAD = find_payload(KAGGLE_INPUT)
if PAYLOAD is not None:
    WORKING = pathlib.Path("/kaggle/working")
else:
    # Not on Kaggle: walk up from the notebook to the repo root.
    PAYLOAD = pathlib.Path.cwd().resolve()
    while PAYLOAD != PAYLOAD.parent:
        if (PAYLOAD / "netra" / "models" / "train.py").exists():
            break
        PAYLOAD = PAYLOAD.parent
    WORKING = PAYLOAD
    print("NOT on Kaggle - using the local repo. No real run will happen here.")

if str(PAYLOAD) not in sys.path:
    sys.path.insert(0, str(PAYLOAD))

ARTIFACTS = WORKING / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
COMMITTED_SPLITS = PAYLOAD / "data" / "splits"
COMMITTED_MANIFEST = PAYLOAD / "data" / "manifest.csv"
DR_CHECKPOINT = ARTIFACTS / "convnext_tiny_aptos.pt"
QUALITY_HEAD = ARTIFACTS / "quality_head_eyeq.pkl"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("payload :", PAYLOAD)
print("working :", WORKING)
print("torch   :", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("gpu     :", torch.cuda.get_device_name(0))

if DEVICE != "cuda" and not ALLOW_CPU:
    raise Stop(
        "STOP: no GPU. Settings -> Accelerator -> GPU T4 x2, then Run All again. "
        "Set ALLOW_CPU = True only to dry-run the plumbing."
    )

if KAGGLE_INPUT.exists():
    print("\nattached inputs:")
    for mount in sorted(KAGGLE_INPUT.iterdir()):
        print("  ", mount)

# A Kaggle Dataset is versioned by hand, so the attached payload is easily an
# older upload than the notebook. That is checked here, where the message can
# say so, rather than surfacing as an AttributeError three cells down.
import importlib
import inspect

REQUIRED = {
    "netra.models.splits": ("find_aptos_label_csvs", "check_preserved", "write_splits"),
    "netra.models.eyeq": ("fetch_label_csvs", "find_eyepacs_roots", "gate_report"),
    "netra.models.dataset": ("find_images_dir", "resolve_image_path"),
    "netra.models.quality": ("QualityGate", "QuickQualFeatures"),
    "netra.models.train": ("train", "save"),
}
stale = []
for module_name, names in REQUIRED.items():
    try:
        module = importlib.import_module(module_name)
    except ImportError as exc:
        stale.append(f"{module_name}: {exc}")
        continue
    stale += [f"{module_name}.{n}" for n in names if not hasattr(module, n)]

import netra.models.train as _train

if "splits_dir" not in inspect.signature(_train.train).parameters:
    stale.append("netra.models.train.train(splits_dir=...)")

if stale:
    raise Stop(
        "STOP: the attached payload is older than this notebook. Missing:\n  "
        + "\n  ".join(stale)
        + "\n\nRebuild it with `python scripts/package_for_kaggle.py`, upload "
        "kaggle_payload.zip as a NEW VERSION of the dataset, and re-attach it. "
        "Without train(splits_dir=...) in particular, a grown run would read "
        "the 366-image manifest while believing it chose the grown one."
    )
print(f"\npayload is current: {len(REQUIRED)} modules expose what this notebook uses")

### 1b. Pinned versions against what Kaggle actually installed

`requirements.txt` says what the numbers in this repo were produced with.
Kaggle's image will not match it exactly and that is fine — but an unnoticed
mismatch in `scikit-learn` or `mapie` is how a pickle stops loading three
weeks later, so the drift is printed rather than assumed away.


In [ ]:
import importlib.metadata as md

PINNED = {}
for line in (PAYLOAD / "requirements.txt").read_text().splitlines():
    line = line.split("#")[0].strip()
    if "==" in line:
        name, _, version = line.partition("==")
        PINNED[name.strip()] = version.strip()

WATCH = ("torch", "timm", "numpy", "scipy", "scikit-learn", "mapie")
print(f"{'package':<16}{'pinned':<14}{'installed':<14}")
VERSIONS = {}
for name in WATCH:
    try:
        installed = md.version(name)
    except md.PackageNotFoundError:
        installed = "MISSING"
    VERSIONS[name] = installed
    flag = "" if installed == PINNED.get(name) else "   <- differs"
    print(f"{name:<16}{PINNED.get(name, '-'):<14}{installed:<14}{flag}")

if VERSIONS.get("mapie") == "MISSING":
    print("\nmapie is absent; section 6 will install it. Everything else is present.")

## 2. Grow the frozen splits to the whole APTOS corpus

The committed splits cover 366 images — one mirror's `test.csv`, which was all
a CPU build box could fetch per file. 219 training images cannot reach the QWK
of 0.88–0.92 that SPEC.md §2 expects, and the full 3,662 are mounted here.

**Growing is not re-shuffling.** `assign_fold` is a pure function of
`(seed, patient_id)`, so an image's fold does not depend on what else is in
the manifest. Every image that already had a fold keeps it — and that is
*checked*: `check_preserved` raises `SplitDrift` and this cell writes nothing
if a single image would move or disappear.

Set `GROW_SPLITS = False` in section 1 to train on the committed 366 instead.


In [ ]:
from netra.models import splits as S

OLD_SPLITS = S.read_splits(COMMITTED_SPLITS)
print("committed:", {k: len(v) for k, v in OLD_SPLITS.items()})

if GROW_SPLITS:
    label_files = S.find_aptos_label_csvs(KAGGLE_INPUT)
    print(
        f"\nAPTOS label files found by header (id_code,diagnosis): {len(label_files)}"
    )
    for path in label_files:
        print("  ", path)
    if not label_files:
        raise Stop(
            "STOP: no APTOS label CSV under /kaggle/input. Attach an APTOS "
            "dataset, or set GROW_SPLITS = False to use the committed 366."
        )

    groups = []
    for path in label_files:
        rows, report = S.read_aptos_labels(path)
        print(json.dumps(report, indent=2))
        groups.append(rows)
    FULL_ROWS = S.merge_manifest_rows(groups)
    print(f"\nmerged to {len(FULL_ROWS)} unique labelled images")

    if COMMITTED_MANIFEST.exists():
        agreement = S.check_labels_agree(
            S.read_manifest_rows(COMMITTED_MANIFEST), FULL_ROWS
        )
        print("no image changed label:", json.dumps(agreement))
    else:
        print("no shipped manifest to cross-check labels against (it is gitignored)")

    NEW_SPLITS, PATIENT_FOLD = S.assign_many(FULL_ROWS)
    PRESERVED = S.check_preserved(OLD_SPLITS, NEW_SPLITS)  # raises SplitDrift
    print("\nnothing moved:")
    print(json.dumps(PRESERVED, indent=2))

    RUN_MANIFEST = ARTIFACTS / "manifest_full.csv"
    RUN_SPLITS = ARTIFACTS / "splits_full"
    digest = S.write_manifest(FULL_ROWS, RUN_MANIFEST)
    SPLIT_META = S.write_splits(
        NEW_SPLITS, RUN_SPLITS, manifest_sha256=digest, patient_fold=PATIENT_FOLD
    )
    print(json.dumps(SPLIT_META, indent=2))
    print(
        "\nCOMMIT artifacts/splits_full/ and artifacts/manifest_full.csv after the run."
    )
else:
    RUN_MANIFEST = COMMITTED_MANIFEST
    RUN_SPLITS = COMMITTED_SPLITS
    PRESERVED = {"n_preserved": sum(len(v) for v in OLD_SPLITS.values()), "n_added": 0}
    SPLIT_META = json.loads((COMMITTED_SPLITS / "meta.json").read_text())
    print("\nGROW_SPLITS is False - training on the committed 366 images.")
    print("Expect QWK well below the 0.88-0.92 in SPEC.md section 2. That is the")
    print("corpus size, not the model.")

## 3. Find the images, and count what is actually there

`.claude/rules/ml.md`: *report the retained fraction at every filtering step —
"reasonable" preprocessing has been documented to silently discard 80–94% of
public corpora.* A split of 2,200 ids with 40 files on disk is a 40-image
experiment, and nothing downstream says so.

This cell stops the run if the train or calibration fraction falls below 0.95.
**The test split is deliberately not collected.**


In [ ]:
from netra.models.dataset import collect, find_images_dir, read_split

train_ids = read_split("train", RUN_SPLITS)
print(f"looking for {len(train_ids)} training images under {KAGGLE_INPUT} ...")
candidates = find_images_dir(train_ids, KAGGLE_INPUT, min_hits=20)
for directory, hits in candidates[:5]:
    print(f"  {hits:>6} hits  {directory}")
if not candidates:
    raise Stop(
        "STOP: no directory under /kaggle/input holds the images named in the "
        "frozen splits. Attach an APTOS image dataset."
    )

IMAGES_DIR = candidates[0][0]
print("\nusing:", IMAGES_DIR)

train_examples, train_report = collect(
    "train", IMAGES_DIR, splits_dir=RUN_SPLITS, manifest=RUN_MANIFEST
)
calib_examples, calib_report = collect(
    "calib", IMAGES_DIR, splits_dir=RUN_SPLITS, manifest=RUN_MANIFEST
)
print(json.dumps({"train": train_report, "calib": calib_report}, indent=2))

for name, report in (("train", train_report), ("calib", calib_report)):
    if report["retained_fraction"] < MIN_RETAINED:
        raise Stop(
            f"STOP: {name} retained {report['retained_fraction']:.4f}, below "
            f"{MIN_RETAINED}. {report['missing_image_file']} files are missing from "
            f"{IMAGES_DIR} and {report['missing_manifest_row']} ids have no manifest "
            "row. Fix the mount before training - the experiment is smaller than "
            "it looks."
        )
print(f"\nboth folds retained >= {MIN_RETAINED}. The TEST split was not touched.")

### 3b. What the crop throws away

Background removal is a filtering step too, so it gets counted. A
`retina_fraction` near π/4 ≈ 0.785 is a full fundus disc inscribed in a
square; far below that is a clipped field of view, and `crop_failed` means
nothing in the frame cleared the background cutoff at all.


In [ ]:
import numpy as np

from netra.models.preprocess import load_image, remove_background

sample = train_examples[:: max(len(train_examples) // 60, 1)][:60]
kept, retina, failed = [], [], 0
for example in sample:
    _, crop = remove_background(load_image(example.path))
    kept.append(crop.retained_fraction)
    retina.append(crop.retina_fraction)
    failed += int(crop.crop_failed)

print(f"crop sampled on {len(sample)} training images")
print(f"  frame kept     median {np.median(kept):.4f}  min {min(kept):.4f}")
print(f"  retina         median {np.median(retina):.4f}  min {min(retina):.4f}")
print(f"  crop failed    {failed}/{len(sample)}")
print("\nNo image is dropped by the crop - it only resizes what it found.")

## 4. Train the DR backbone

`convnext_tiny.in12k_ft_in1k` at 512px, 5 classes.

Model selection is on the **calibration** fold and nothing else. A checkpoint
is written only when calibration QWK improves, and it carries `trained: true`
— which is what `DRClassifier` reads to decide whether these weights may be
reported as a model at all. A `--smoke` artifact carries `trained: false` and
reports itself as `:untrained` forever.

This is the long cell: roughly 1.5–3 hours on a T4 at the default 12 epochs.


In [ ]:
from netra.models.train import train

started = time.time()
result = train(
    images_dir=IMAGES_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    device=DEVICE,
    smoke=False,
    out=DR_CHECKPOINT,
    pretrained=True,
    splits_dir=RUN_SPLITS,
    manifest=RUN_MANIFEST,
)
print(json.dumps({k: v for k, v in result.items() if k != "history"}, indent=2))
print(f"\nwall clock: {(time.time() - started) / 60:.1f} min")

## 5. Calibration QWK, and the leak alarm

The red line is not a target. SPEC.md §2: **anything above 0.95 on a local
split means a leaked split**, and the next cell stops the run rather than
letting a leaked number reach a slide.


In [ ]:
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in result["history"]]
qwk = [h.get("calib", {}).get("qwk") for h in result["history"]]

plt.figure(figsize=(6, 3.5))
plt.plot(epochs, qwk, marker="o")
plt.axhline(LEAK_ALARM_QWK, ls="--", lw=1, color="crimson")
plt.axhspan(0.88, 0.92, color="seagreen", alpha=0.12)
plt.annotate(
    "above this: suspect a leaked split",
    (epochs[0], LEAK_ALARM_QWK + 0.005),
    color="crimson",
    fontsize=8,
)
plt.annotate(
    "SPEC.md expects 0.88-0.92", (epochs[0], 0.90), color="seagreen", fontsize=8
)
plt.xlabel("epoch")
plt.ylabel("calibration QWK")
plt.ylim(0, 1)
plt.title("Model selection on the CALIBRATION fold")
plt.tight_layout()
plt.show()

In [ ]:
best = result["best_calib"]
print("best calibration epoch:", json.dumps(best))

if best is None or best["qwk"] is None or best["qwk"] < 0:
    raise Stop(
        "STOP: no epoch produced a measurable calibration QWK, so no checkpoint "
        "was written. There is no model to report."
    )

if best["qwk"] > LEAK_ALARM_QWK:
    raise Stop(
        f"STOP: calibration QWK {best['qwk']:.4f} is above {LEAK_ALARM_QWK}. "
        "SPEC.md section 2: that means a LEAKED SPLIT, not a good model. APTOS "
        "publishes no patient id, so two images of the same patient can sit in "
        "different folds and nothing here would catch it. Find the leak before "
        "quoting this number anywhere."
    )

if not (0.88 <= best["qwk"] <= 0.92):
    print(
        f"\nNOTE: {best['qwk']:.4f} is outside the 0.88-0.92 SPEC.md section 2 "
        "expects. Report it as measured; do not tune against it."
    )

DR_PROVENANCE = {
    "artifact": str(DR_CHECKPOINT),
    "real": True,
    "backbone": result["backbone"],
    "image_size": 512,
    "epochs_run": result["epochs_run"],
    "best_calib": best,
    "selected_on": "calibration split only",
    "test_split_touched": False,
    "splits": SPLIT_META,
    "split_growth": PRESERVED,
    "train_report": result["train_report"],
    "calib_report": result["calib_report"],
    "images_dir": str(IMAGES_DIR),
    "versions": VERSIONS,
    "history": result["history"],
}
(ARTIFACTS / "dr_run.json").write_text(json.dumps(DR_PROVENANCE, indent=2))
print("\nwrote", ARTIFACTS / "dr_run.json")

## 6. Conformal abstention, fitted on the calibration fold

MAPIE's `SplitConformalClassifier` with `conformity_score="lac"` and
`prefit=True`.

The abstention rule is not "the set is large" — it is "the set spans more than
one **clinical action**". `{0,1}` and `{0,3}` are both size 2 and only the
second is alarming.


In [ ]:
try:
    import mapie  # noqa: F401
except ImportError:
    pinned = PINNED.get("mapie", "1.6.0")
    print(f"installing mapie=={pinned} ...")
    import subprocess

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"mapie=={pinned}"], check=True
    )

from netra.models.abstain import AbstentionWrapper, PrefitProbaEstimator
from netra.models.dataset import load_batch
from netra.models.dr import DRClassifier

clf = DRClassifier(checkpoint=DR_CHECKPOINT, device=DEVICE, pretrained_backbone=False)
if not clf.trained:
    raise Stop(
        "STOP: refusing to conformalise weights that are not marked trained:true. "
        f"model_version is {clf.model_version!r}."
    )
print("conformalising", clf.model_version)

# Chunked so the whole calibration fold is never resident as raw pixels at once.
proba_rows, label_rows = [], []
for start in range(0, len(calib_examples), 64):
    chunk = calib_examples[start : start + 64]
    images, labels = load_batch(chunk)
    proba_rows.extend(clf.predict_proba(img) for img in images)
    label_rows.extend(labels.tolist())
    print(f"  {min(start + 64, len(calib_examples))}/{len(calib_examples)}", end="\r")

calib_proba = np.stack(proba_rows)
calib_labels = np.asarray(label_rows)
print(f"\ncalibration probabilities: {calib_proba.shape}")

wrapper = AbstentionWrapper(PrefitProbaEstimator(n_classes=5))
wrapper.conformalize(calib_proba, calib_labels)
CONFORMAL = wrapper.coverage_report(calib_proba, calib_labels)
print(json.dumps(CONFORMAL, indent=2))

(ARTIFACTS / "conformal_calibration.json").write_text(
    json.dumps(
        {
            "real": True,
            "fitted_on": "calibration split only",
            "n_calibration": len(calib_labels),
            "confidence_level": wrapper.confidence_level,
            "model_version": clf.model_version,
            "report": CONFORMAL,
        },
        indent=2,
    )
)
print("\nwrote", ARTIFACTS / "conformal_calibration.json")

## 7. The REAL quality head — EyeQ labels on the mounted EyePACS

This is what replaces the synthetic stand-in. EyeQ (Fu et al., MICCAI 2019)
publishes 28,792 three-level gradability judgements made by human graders, and
no pixels — its images are EyePACS's, which is why this was blocked on a CPU
box and is not blocked here.

* Labels: two CSVs from `github.com/HzFu/EyeQ`, ~700 KiB. **Internet must be On.**
* Images: the attached EyePACS mount. **Nothing is downloaded.**

The head is binary — Good+Usable against Reject — because that is the decision
the gate makes, and the setting QuickQual's published 0.886 accuracy / 0.9687
AUC refers to. The three-way `usable | borderline | reject` verdict is cut
downstream from `config/thresholds.yaml`.

### 7a. Fetch the labels


In [ ]:
from netra.models import eyeq

EYEQ_DIR = ARTIFACTS / "eyeq_labels"
try:
    label_csvs = eyeq.fetch_label_csvs(EYEQ_DIR)
except Exception as exc:
    raise Stop(
        f"STOP: could not fetch the EyeQ labels ({type(exc).__name__}: {exc}). "
        "Settings -> Internet -> On. Only these two CSVs need the network; no "
        "images are downloaded."
    ) from exc

EYEQ_ROWS, EYEQ_REPORTS = {}, {}
for name, path in label_csvs.items():
    rows, report = eyeq.parse_label_csv(path)
    EYEQ_ROWS[name] = rows
    EYEQ_REPORTS[name] = report
    print(json.dumps(report, indent=2))

train_patients = {r.patient_id for r in EYEQ_ROWS["train"]}
test_patients = {r.patient_id for r in EYEQ_ROWS["test"]}
overlap = train_patients & test_patients
print(f"\npublished EyeQ folds: {len(train_patients)} / {len(test_patients)} patients")
print(f"patients in both: {len(overlap)}")
assert not overlap, "the published EyeQ folds are not patient-disjoint after all"

### 7b. Resolve the labels against the mounted EyePACS

The number to read first is the retained fraction. EyeQ's train rows come from
the EyePACS **train** set and its test rows from the EyePACS **test** set, so
a mirror carrying only one of the two resolves roughly half the labels — which
is normal and is *not* the 0.95 stop that applies to the APTOS folds. What
matters here is that both folds end up with enough of both classes to fit and
to measure.


In [ ]:
print(f"looking for EyePACS-named images under {KAGGLE_INPUT} ...")
eyepacs_roots = eyeq.find_eyepacs_roots(KAGGLE_INPUT, min_files=200)
for directory, count in eyepacs_roots[:6]:
    print(f"  {count:>7} images  {directory}")
if not eyepacs_roots:
    raise Stop(
        "STOP: no directory under /kaggle/input holds <patient>_<left|right> "
        "images. Attach an EyePACS mirror (it is mounted, not downloaded)."
    )

EYEPACS_DIRS = [d for d, _ in eyepacs_roots]
INDEX = eyeq.index_images(EYEPACS_DIRS)
print(
    f"\nindexed {len(INDEX)} EyePACS images across {len(EYEPACS_DIRS)} directory(ies)"
)

RESOLVED, RESOLVE_REPORTS = {}, {}
for name, rows in EYEQ_ROWS.items():
    pairs, report = eyeq.resolve(rows, INDEX)
    RESOLVED[name] = pairs
    RESOLVE_REPORTS[name] = report
    print(f"\nEyeQ {name}:")
    print(json.dumps(report, indent=2))

### 7c. Folds

If the mount resolves both published EyeQ folds, those are used — a published
split beats one we invent. If it only reaches one of them, everything resolved
is pooled and split **patient-level** by the same seeded hash the APTOS splits
use, and the cell says which path it took. Either way both eyes of a patient
stay together, and that is asserted rather than assumed.


In [ ]:
MIN_PER_FOLD = 200
have_both = all(len(RESOLVED[n]) >= MIN_PER_FOLD for n in ("train", "test"))

if have_both:
    FOLDS = {"fit": RESOLVED["train"], "eval": RESOLVED["test"]}
    FOLD_SOURCE = "EyeQ's own published train/test folds"
else:
    pooled = RESOLVED["train"] + RESOLVED["test"]
    FOLDS = eyeq.hash_holdout(pooled)
    FOLD_SOURCE = (
        "a patient-level hash holdout over everything the mount resolved "
        "(the published EyeQ folds were not both reachable on this mount)"
    )
print("folds from:", FOLD_SOURCE)

assert eyeq.patient_disjoint(FOLDS), "a patient reached both folds"

CAP_REPORTS = {}
for name in ("fit", "eval"):
    FOLDS[name], CAP_REPORTS[name] = eyeq.cap_per_class(FOLDS[name], EYEQ_PER_CLASS)
    print(f"\n{name}: {json.dumps(CAP_REPORTS[name])}")

assert eyeq.patient_disjoint(FOLDS), "capping moved a patient across folds"

for name in ("fit", "eval"):
    y = eyeq.labels_for(FOLDS[name])
    if len(set(y.tolist())) < 2:
        raise Stop(
            f"STOP: the {name} fold has only one class after resolving against "
            "this mount. A one-class head answers every image the same way, "
            "confidently. Attach a fuller EyePACS mirror."
        )
    print(f"{name}: n={len(y)}  gradable={int(y.sum())}  reject={int((1 - y).sum())}")

### 7d. Features and fit

`densenet121.tv_in1k` is **frozen** — that is the whole point of QuickQual and
why only a small head is fitted. The backbone's real pretrained weights are
used here; a `random-init` run would produce a head whose `model_version` says
so, and nothing would report it as real.


In [ ]:
from netra.models.quality import QualityGate, QuickQualFeatures

features = QuickQualFeatures(pretrained=True, device=DEVICE)
print("features:", features.version)

FEATURE_REPORTS, X, Y = {}, {}, {}
for name in ("fit", "eval"):
    started = time.time()
    paths = [p for _, p in FOLDS[name]]
    matrix, kept_idx, report = eyeq.features_for_paths(
        features, paths, batch_size=EYEQ_BATCH_SIZE
    )
    FOLDS[name] = [FOLDS[name][i] for i in kept_idx]
    X[name] = matrix
    Y[name] = eyeq.labels_for(FOLDS[name])
    FEATURE_REPORTS[name] = report
    print(f"\n{name}: {json.dumps(report, indent=2)}")
    print(f"  {matrix.shape} in {(time.time() - started) / 60:.1f} min")

gate = QualityGate(features=features, head=None)
gate.fit_features(X["fit"], Y["fit"])
print("\nfitted head:", type(gate.head).__name__, "|", gate.model_version)

### 7e. Measure it, then save it

Every number comes from `netra/eval/metrics.py`. Accuracy travels with its
Wilson interval — never a bare percentage (CLAUDE.md). The verdict table
counts the failure that actually matters on its own: an **ungradable image
admitted as `usable`**, which is the one that lets a bad capture become a
grade.


In [ ]:
p_eval = np.asarray([gate._p_usable(row) for row in X["eval"]])
GATE_REAL = eyeq.gate_report(Y["eval"], p_eval)
VERDICTS_REAL = eyeq.verdict_report(gate, p_eval, Y["eval"])

print("REAL EyeQ labels - held-out fold")
print(json.dumps(GATE_REAL, indent=2))
print("\nverdicts at the thresholds in config/thresholds.yaml")
print(json.dumps(VERDICTS_REAL, indent=2))
print(
    "\nPublished QuickQual on EyeQ: accuracy 0.886, AUC 0.9687, kappa 0.811. "
    "Those are the PAPER's numbers on the paper's setup, not ours - do not "
    "print them beside the table above as if they were the same measurement."
)

import pickle

with QUALITY_HEAD.open("wb") as fh:
    pickle.dump(gate.head, fh)

QUALITY_PROVENANCE = {
    "artifact": str(QUALITY_HEAD),
    "real": True,
    "label_source": "EyeQ (HzFu/EyeQ), human three-level gradability grading",
    "label_urls": eyeq.LABEL_URLS,
    "image_source": [str(d) for d in EYEPACS_DIRS],
    "task": "binary: Good+Usable (1) vs Reject (0)",
    "backbone": features.version,
    "head": type(gate.head).__name__,
    "model_version": gate.model_version,
    "fold_source": FOLD_SOURCE,
    "label_parse": EYEQ_REPORTS,
    "resolve_against_mount": RESOLVE_REPORTS,
    "cap_per_class": CAP_REPORTS,
    "feature_pass": FEATURE_REPORTS,
    "n_fit": len(Y["fit"]),
    "n_eval": len(Y["eval"]),
    "metrics_on_held_out": GATE_REAL,
    "verdicts_on_held_out": VERDICTS_REAL,
    "versions": VERSIONS,
    "not_comparable_to": "data/quality_head_synthetic.pkl (synthetic degradations)",
}
(ARTIFACTS / "quality_head_eyeq.json").write_text(
    json.dumps(QUALITY_PROVENANCE, indent=2)
)
print("\nwrote", QUALITY_HEAD)
print("wrote", ARTIFACTS / "quality_head_eyeq.json")

## 8. What is real, what is synthetic, and where it all landed

`.claude/rules/ml.md`: *real poor-quality images and synthetic degradations go
in SEPARATE tables. They are different distributions and pooling them is a
validity error.* The two tables below are never added together.


In [ ]:
line = "-" * 78
print(line)
print("REAL - measured on real data in this run")
print(line)
print(f"  DR backbone        {result['backbone']} @512, {result['epochs_run']} epochs")
folds = " / ".join(f"{k} {v}" for k, v in SPLIT_META["n_images"].items())
print(f"  DR corpus          APTOS - {folds}")
print(f"  calibration QWK    {best['qwk']:.4f}   (calibration fold only)")
print(
    f"  conformal          coverage {CONFORMAL['coverage']:.4f}, abstention "
    f"{CONFORMAL['abstention_rate']:.4f} at {wrapper.confidence_level} "
    "(calibration fold)"
)
print(
    f"  quality labels     EyeQ, {sum(r['n_kept'] for r in EYEQ_REPORTS.values())} "
    "human gradability judgements"
)
print(f"  quality fold       {FOLD_SOURCE}")
n, lo_hi = GATE_REAL["n"], GATE_REAL["accuracy_ci"]
print(
    f"  gate accuracy      {GATE_REAL['n_correct']}/{n} = {GATE_REAL['accuracy']:.4f} "
    f"[{lo_hi[0]:.4f}, {lo_hi[1]:.4f}] Wilson"
)
print(f"  gate Brier / NLL   {GATE_REAL['brier']:.4f} / {GATE_REAL['nll']:.4f}")
print(
    f"  ungradable let in  {VERDICTS_REAL['ungradable_admitted_as_usable']} images "
    "called 'usable' that a human called Reject"
)

print()
print(line)
print("SYNTHETIC - separate table, never pooled with the above")
print(line)
print("  data/quality_head_synthetic.pkl - blur / under-exposure / clipped-FOV")
print("  degradations of real images. A human never judged them. It is superseded")
print("  by quality_head_eyeq.pkl and kept only so the two can be compared as what")
print("  they are: two different distributions.")

print()
print(line)
print("NOT MEASURED - do not quote")
print(line)
print("  Anything on the TEST split. It was not collected anywhere in this run,")
print("  and it is touched exactly once, at the very end, outside this notebook.")
print("  The paper's QuickQual numbers (0.886 / 0.9687 / 0.811) are the paper's.")

SUMMARY = {
    "real": {
        "dr_checkpoint": str(DR_CHECKPOINT),
        "dr_calib_qwk": best["qwk"],
        "quality_head": str(QUALITY_HEAD),
        "quality_accuracy": GATE_REAL["accuracy"],
        "quality_accuracy_ci": GATE_REAL["accuracy_ci"],
        "quality_label_source": "EyeQ human grading",
    },
    "synthetic": {
        "quality_head": "data/quality_head_synthetic.pkl",
        "label_source": "cv2 degradations; superseded by the EyeQ head",
    },
    "not_measured": ["test split", "any number on real poor-quality APTOS images"],
    "artifacts": sorted(str(p.name) for p in ARTIFACTS.rglob("*") if p.is_file()),
}
(ARTIFACTS / "run_summary.json").write_text(json.dumps(SUMMARY, indent=2))

print()
print(line)
print(f"ARTIFACTS - download these from the Output tab ({ARTIFACTS})")
print(line)
for path in sorted(ARTIFACTS.rglob("*")):
    if path.is_file():
        print(
            f"  {path.relative_to(ARTIFACTS).as_posix():<34}{path.stat().st_size:>12,} B"
        )

print()
print("Next, off Kaggle:")
print("  1. commit artifacts/splits_full/ and artifacts/manifest_full.csv")
print("  2. NETRA_DR_CHECKPOINT=<convnext_tiny_aptos.pt> \\")
print("     NETRA_QUICKQUAL_SVM=<quality_head_eyeq.pkl> \\")
print("     python -m netra.models.fit_thresholds --images <dir>")
print("  3. the TEST split, once, and only then.")